In [1]:
%pip show osmnx
%pip show matplotlib
%pip show contextily
%pip show folium

Name: osmnx
Version: 2.1.0
Summary: Download, model, analyze, and visualize street networks and other geospatial features from OpenStreetMap
Home-page: 
Author: Geoff Boeing
Author-email: Geoff Boeing <boeing@usc.edu>
License: 
Location: /opt/conda/lib/python3.11/site-packages
Requires: geopandas, networkx, numpy, pandas, requests, shapely
Required-by: 
Note: you may need to restart the kernel to use updated packages.
Name: matplotlib
Version: 3.8.3
Summary: Python plotting package
Home-page: https://matplotlib.org
Author: John D. Hunter, Michael Droettboom
Author-email: matplotlib-users@python.org
License: PSF
Location: /opt/conda/lib/python3.11/site-packages
Requires: contourpy, cycler, fonttools, kiwisolver, numpy, packaging, pillow, pyparsing, python-dateutil
Required-by: contextily, ipympl, seaborn
Note: you may need to restart the kernel to use updated packages.
Name: contextily
Version: 1.7.0
Summary: Context geo-tiles in Python
Home-page: 
Author: 
Author-email: Dani Arribas-Be

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType ,StructType, StructField

import folium
from folium.plugins import HeatMap

import osmnx as ox
import matplotlib.pyplot as plt

import folium
from folium.plugins import FastMarkerCluster, HeatMap

# Initialize the SparkSession 

In [3]:
spark = SparkSession.builder \
    .appName("PortoTripsAnalysis") \
    .getOrCreate()

In [4]:
spark

In [5]:
df = spark.read.csv("s3a://taasim/raw/porto-trips/train.csv", header=True)

In [6]:
df.show(10)

+-------------------+---------+-----------+------------+--------+----------+--------+------------+--------------------+
|            TRIP_ID|CALL_TYPE|ORIGIN_CALL|ORIGIN_STAND| TAXI_ID| TIMESTAMP|DAY_TYPE|MISSING_DATA|            POLYLINE|
+-------------------+---------+-----------+------------+--------+----------+--------+------------+--------------------+
|1372636858620000589|        C|       NULL|        NULL|20000589|1372636858|       A|       False|[[-8.618643,41.14...|
|1372637303620000596|        B|       NULL|           7|20000596|1372637303|       A|       False|[[-8.639847,41.15...|
|1372636951620000320|        C|       NULL|        NULL|20000320|1372636951|       A|       False|[[-8.612964,41.14...|
|1372636854620000520|        C|       NULL|        NULL|20000520|1372636854|       A|       False|[[-8.574678,41.15...|
|1372637091620000337|        C|       NULL|        NULL|20000337|1372637091|       A|       False|[[-8.645994,41.18...|
|1372636965620000231|        C|       NU

In [7]:
df.describe()

DataFrame[summary: string, TRIP_ID: string, CALL_TYPE: string, ORIGIN_CALL: string, ORIGIN_STAND: string, TAXI_ID: string, TIMESTAMP: string, DAY_TYPE: string, MISSING_DATA: string, POLYLINE: string]

In [8]:
df.select("TRIP_ID", "TAXI_ID", "POLYLINE").show(5)

+-------------------+--------+--------------------+
|            TRIP_ID| TAXI_ID|            POLYLINE|
+-------------------+--------+--------------------+
|1372636858620000589|20000589|[[-8.618643,41.14...|
|1372637303620000596|20000596|[[-8.639847,41.15...|
|1372636951620000320|20000320|[[-8.612964,41.14...|
|1372636854620000520|20000520|[[-8.574678,41.15...|
|1372637091620000337|20000337|[[-8.645994,41.18...|
+-------------------+--------+--------------------+
only showing top 5 rows



In [9]:
df.count()

1710670

# Parsing the Coordinates

In [10]:
# df.select("POLYLINE").show(1,truncate=False)

In [11]:
# Define the schema for the GPS points
gps_schema = ArrayType(ArrayType(DoubleType()))
# Convert the string column to an actual Array of coordinates
df_parsed = df.withColumn("coords", F.from_json(F.col("POLYLINE"), gps_schema))
# Explode the list so each GPS ping becomes its own row
df_exploded = df_parsed.withColumn("point", F.explode(F.col("coords")))

In [12]:
# df_exploded.show(5 ,truncate=False)

In [13]:
# df_exploded.show(5,truncate=False)
df_exploded.dtypes

[('TRIP_ID', 'string'),
 ('CALL_TYPE', 'string'),
 ('ORIGIN_CALL', 'string'),
 ('ORIGIN_STAND', 'string'),
 ('TAXI_ID', 'string'),
 ('TIMESTAMP', 'string'),
 ('DAY_TYPE', 'string'),
 ('MISSING_DATA', 'string'),
 ('POLYLINE', 'string'),
 ('coords', 'array<array<double>>'),
 ('point', 'array<double>')]

# The Transformation Math

In [14]:
# P_LON_MIN, P_LON_MAX = -8.7, -8.5
# P_LAT_MIN, P_LAT_MAX = 41.1, 41.2

# C_LON_MIN, C_LON_MAX = -7.8, -7.4
# C_LAT_MIN, C_LAT_MAX = 33.4, 33.7

In [15]:
def transform_coordinates(porto_lon, porto_lat):
    """
    Linearly maps Porto GPS coordinates to Casablanca coordinates.
    Input: (lon, lat) in Porto
    Output: (cas_lon, cas_lat) in Casablanca
    """
    # 1. Define Bounding Box Constants
    P_LON_MIN, P_LON_MAX = -8.7, -8.5
    P_LAT_MIN, P_LAT_MAX = 41.1, 41.2

    C_LON_MIN, C_LON_MAX = -7.8, -7.4
    C_LAT_MIN, C_LAT_MAX = 33.4, 33.7

    # 2. Linear Transformation for Longitude
    # (Value - Min) / (Max - Min) * Range + New_Min
    cas_lon = C_LON_MIN + (porto_lon - P_LON_MIN) / (P_LON_MAX - P_LON_MIN) * (C_LON_MAX - C_LON_MIN)

    # 3. Linear Transformation for Latitude
    cas_lat = C_LAT_MIN + (porto_lat - P_LAT_MIN) / (P_LAT_MAX - P_LAT_MIN) * (C_LAT_MAX - C_LAT_MIN)

    return cas_lon, cas_lat


# 1. Define the return schema (a pair of doubles)
schema = StructType([
    StructField("cas_lon", DoubleType(), False),
    StructField("cas_lat", DoubleType(), False)
])

# User Defined Function
# 2. Register the function as a Spark UDF
transform_udf = F.udf(transform_coordinates, schema)


In [16]:
# Task 04 — boundary assertions for the linear transform

def _assert_close(a: float, b: float, tol: float = 1e-9) -> None:
    assert abs(a - b) <= tol, f"{a} != {b} (tol={tol})"


P_LON_MIN, P_LON_MAX = -8.7, -8.5
P_LAT_MIN, P_LAT_MAX = 41.1, 41.2

C_LON_MIN, C_LON_MAX = -7.8, -7.4
C_LAT_MIN, C_LAT_MAX = 33.4, 33.7

cases = [
    ((P_LON_MIN, P_LAT_MIN), (C_LON_MIN, C_LAT_MIN)),
    ((P_LON_MIN, P_LAT_MAX), (C_LON_MIN, C_LAT_MAX)),
    ((P_LON_MAX, P_LAT_MIN), (C_LON_MAX, C_LAT_MIN)),
    ((P_LON_MAX, P_LAT_MAX), (C_LON_MAX, C_LAT_MAX)),
    (
        ((P_LON_MIN + P_LON_MAX) / 2.0, (P_LAT_MIN + P_LAT_MAX) / 2.0),
        ((C_LON_MIN + C_LON_MAX) / 2.0, (C_LAT_MIN + C_LAT_MAX) / 2.0),
    ),
]

for (porto_lon, porto_lat), (exp_lon, exp_lat) in cases:
    cas_lon, cas_lat = transform_coordinates(porto_lon, porto_lat)
    _assert_close(float(cas_lon), float(exp_lon))
    _assert_close(float(cas_lat), float(exp_lat))

print("transform_coordinates boundary tests: OK")


transform_coordinates boundary tests: OK


In [17]:
# 3. Apply it to your exploded DataFrame
# 1. Extract raw Porto lon/lat from the 'point' array column
df_with_porto = df_exploded.withColumn("porto_lon", F.col("point")[0]) \
                           .withColumn("porto_lat", F.col("point")[1])
df_with_porto = df_with_porto.filter(
    (F.col("porto_lon") >= -8.7) & (F.col("porto_lon") <= -8.5) &
    (F.col("porto_lat") >= 41.1) & (F.col("porto_lat") <= 41.2)
)

# 2. Run the transformation UDF
# This creates a struct column called 'transformed' containing both new coords
df_transformed = df_with_porto.withColumn("transformed", transform_udf(F.col("porto_lon"), F.col("porto_lat")))

# 3. Flatten the struct and select our final columns
df_final = df_transformed.select(
    "TRIP_ID", 
    "TAXI_ID", 
    "TIMESTAMP",
    "CALL_TYPE",
    "transformed.cas_lon", 
    "transformed.cas_lat"
)


In [18]:
df_final.show(5,truncate=False)

+-------------------+--------+----------+---------+------------------+------------------+
|TRIP_ID            |TAXI_ID |TIMESTAMP |CALL_TYPE|cas_lon           |cas_lat           |
+-------------------+--------+----------+---------+------------------+------------------+
|1372636858620000589|20000589|1372636858|C        |-7.637286000000002|33.524236         |
|1372636858620000589|20000589|1372636858|C        |-7.636998000000001|33.524128         |
|1372636858620000589|20000589|1372636858|C        |-7.640652000000002|33.52753          |
|1372636858620000589|20000589|1372636858|C        |-7.644306000000003|33.531444999999984|
|1372636858620000589|20000589|1372636858|C        |-7.647906000000002|33.533119         |
+-------------------+--------+----------+---------+------------------+------------------+
only showing top 5 rows



In [19]:
# Apply Transformation for both Lon and Lat
# df_final = df_exploded.withColumn("cas_lon", 
#     F.lit(C_LON_MIN) + ( (F.col("point")[0] - P_LON_MIN) / (P_LON_MAX - P_LON_MIN) ) * (C_LON_MAX - C_LON_MIN)
# ).withColumn("cas_lat", 
#     F.lit(C_LAT_MIN) + ( (F.col("point")[1] - P_LAT_MIN) / (P_LAT_MAX - P_LAT_MIN) ) * (C_LAT_MAX - C_LAT_MIN)
# )

In [20]:
df_final.show(5,truncate=False)

+-------------------+--------+----------+---------+------------------+------------------+
|TRIP_ID            |TAXI_ID |TIMESTAMP |CALL_TYPE|cas_lon           |cas_lat           |
+-------------------+--------+----------+---------+------------------+------------------+
|1372636858620000589|20000589|1372636858|C        |-7.637286000000002|33.524236         |
|1372636858620000589|20000589|1372636858|C        |-7.636998000000001|33.524128         |
|1372636858620000589|20000589|1372636858|C        |-7.640652000000002|33.52753          |
|1372636858620000589|20000589|1372636858|C        |-7.644306000000003|33.531444999999984|
|1372636858620000589|20000589|1372636858|C        |-7.647906000000002|33.533119         |
+-------------------+--------+----------+---------+------------------+------------------+
only showing top 5 rows



# Mapping Coordinates to Neighborhoods

In [21]:
# Load the Casablanca Zone Mapping (Metadata)
zones_df = spark.read.csv("s3a://taasim/metadata/zone_mapping.csv", header=True, inferSchema=True)

In [22]:
zones_df.show(5,truncate=False)

+-----------------+------------+-------+-------+-------+-------+
|arrondissement_id|zone_name   |lon_min|lon_max|lat_min|lat_max|
+-----------------+------------+-------+-------+-------+-------+
|1                |Sidi Belyout|-7.625 |-7.595 |33.59  |33.62  |
|2                |Maarif      |-7.66  |-7.615 |33.565 |33.595 |
|3                |Anfa        |-7.71  |-7.66  |33.585 |33.63  |
|4                |Hay Hassani |-7.73  |-7.67  |33.53  |33.585 |
|5                |Mers Sultan |-7.625 |-7.595 |33.555 |33.59  |
+-----------------+------------+-------+-------+-------+-------+
only showing top 5 rows



In [23]:
enriched_df = df_final.join(
    zones_df,
    (df_final.cas_lon >= zones_df.lon_min) & 
    (df_final.cas_lon <= zones_df.lon_max) & 
    (df_final.cas_lat >= zones_df.lat_min) & 
    (df_final.cas_lat <= zones_df.lat_max),
    how="left"
).fillna({"arrondissement_id" : -1}) # 'out_of_bounds' as -1

In [24]:
# enriched_df.dtypes

In [25]:
final_output = enriched_df.select(
    "TRIP_ID", 
    "TAXI_ID", 
    F.col("TIMESTAMP").cast("long"),
    F.col("cas_lon"), 
    F.col("cas_lat"), 
    F.col("CALL_TYPE"),
    F.col("arrondissement_id"), 
    "zone_name"
)

In [26]:
# final_output.show(10)

In [27]:
final_output.dtypes

[('TRIP_ID', 'string'),
 ('TAXI_ID', 'string'),
 ('TIMESTAMP', 'bigint'),
 ('cas_lon', 'double'),
 ('cas_lat', 'double'),
 ('CALL_TYPE', 'string'),
 ('arrondissement_id', 'int'),
 ('zone_name', 'string')]

# Analytics Plots

In [ ]:
# 1. Call-Type Breakdown
call_types = final_output.groupBy("CALL_TYPE").count().toPandas()
call_types.plot(kind='pie', y='count', labels=call_types['CALL_TYPE'], autopct='%1.1f%%', title="Call Type Distribution")

In [ ]:
# 2. Temporal Demand Curve
# Note: F.hour requires a Timestamp type; since your TIMESTAMP is bigint (Unix), we convert it first
demand = final_output.withColumn("hour", F.hour(F.from_unixtime(F.col("TIMESTAMP")))) \
                     .groupBy("hour").count().orderBy("hour").toPandas()
demand.plot(kind='line', x='hour', y='count', title="Trips per Hour of Day")

In [ ]:
# 3. Trip Duration (Approximate: 15s per GPS point)
duration = df_parsed.withColumn("duration_min", (F.size(F.col("coords")) * 15) / 60) \
                    .select("duration_min").toPandas()
duration.plot(kind='hist', bins=50, title="Trip Duration Distribution (Minutes)")

# Validation & Profiling

In [ ]:
# 1. Download the Casablanca map (The "Canvas")
place_name = "Casablanca, Morocco"
graph = ox.graph_from_place(place_name, network_type='drive')

# 2. Convert Spark Data to Pandas for plotting (The "Paint")
# We sample 10k points so the map remains readable
plot_df = final_output.select("cas_lon", "cas_lat").sample(0.01).limit(10000).toPandas()

# 3. Create the Plot
# we set 'show=False' and 'close=False' so we can add the taxis before rendering
fig, ax = ox.plot_graph(graph, node_size=0, edge_color='#cccccc', 
                        edge_linewidth=0.5, show=False, close=False, 
                        figsize=(12, 12))

# 4. Overlay the Taxis
ax.scatter(plot_df['cas_lon'], plot_df['cas_lat'], 
           s=1, color='red', alpha=0.3, label='Transformed Taxis')

# 5. Styling and Saving
ax.set_title("TaaSim: Porto-to-Casablanca Validation Map", fontsize=15, color='white')
plt.savefig(r"../data/casablanca-coordinate-validation.png", facecolor='#111111', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. Prepare the data
plot_df = final_output.select("cas_lon", "cas_lat").sample(0.01).limit(10000).toPandas()

# 2. Create the plot
plt.figure(figsize=(10, 8))
plt.scatter(plot_df['cas_lon'], plot_df['cas_lat'], s=2, alpha=0.5, color='teal')

# 3. Add the title and labels
plt.title("TaaSim: Casablanca Taxi Coordinate Validation")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

# 4. SAVE IT AS A PNG
# This writes the file directly to your 'docs' folder
plt.savefig(r"../data/casablanca-coordinate-validation.png")

print("Success! Check your 'docs' folder for the image.")

In [ ]:
# Define the place name
place_name = "Casablanca, Morocco"

# Download the street network (driving paths)
# This handles the downloading and filtering automatically
graph = ox.graph_from_place(place_name, network_type='drive')

# Plot the map
fig, ax = ox.plot_graph(graph, node_size=0, edge_color='blue', edge_linewidth=0.5)
plt.show()

# To save it as a Shapefile or GeoJSON
# ox.save_graph_geopackage(graph, filepath='casablanca_map.gpkg')

In [ ]:
# # 1. Sample the data 
# # (Interactive maps lag if you use too many points; 5,000 is the sweet spot)
# map_data = final_output.select("cas_lat", "cas_lon").sample(0.01).limit(5000).collect()
# points = [[p.cas_lat, p.cas_lon] for p in map_data]

# # 2. Initialize the Map centered on Casablanca
# # 'tiles' is where we set the Google Maps look
# m = folium.Map(
#     location=[33.5731, -7.5898], 
#     zoom_start=12,
#     tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
#     attr='Google'
# )

# # 3. Add the Taxis as small CircleMarkers
# # This allows you to click/see exactly where they land on the streets
# for point in points:
#     folium.CircleMarker(
#         location=point,
#         radius=2,
#         color='red',
#         fill=True,
#         fill_color='red',
#         fill_opacity=0.7
#     ).add_to(m)

# # 4. Save as HTML (This is the file you can open in any browser)
# m.save(r"../data/casablanca-interactive-validation.html")

# # 5. Display in Jupyter
# m

In [ ]:
# # 1. Prepare your transformed Spark data
# # Sample 5,000 points so the browser doesn't lag
# map_sample = final_output.select("cas_lat", "cas_lon").sample(0.01).limit(5000).collect()
# points = [[p.cas_lat, p.cas_lon] for p in map_sample]

# # 2. Create the Interactive Map
# # We center it on the heart of Casablanca
# m = folium.Map(
#     location=[33.5731, -7.5898], 
#     zoom_start=13,
#     tiles="OpenStreetMap" # This is the standard interactive OSM style
# )

# # 3. Add the Taxis as a "HeatMap"
# # This is better for interactivity than 5,000 separate dots
# HeatMap(points, radius=10, blur=15, gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}).add_to(m)

# # 4. Save and Show
# m.save(r"../data/interactive_validation.html")
# m

In [ ]:
# 1. Define the Casablanca "Lock" (Bounding Box)
# These coordinates match our Task 04 Technical Hints
C_LAT_MIN, C_LAT_MAX = 33.4, 33.7
C_LON_MIN, C_LON_MAX = -7.8, -7.4

# 2. Prepare the data (Sample 5,000 points)
map_sample = final_output.select("cas_lat", "cas_lon").sample(0.01).limit(5000).collect()
points = [[p.cas_lat, p.cas_lon] for p in map_sample]

# 3. Initialize the Map with strict "Founder B" constraints
m = folium.Map(
    location=[33.5731, -7.5898], 
    zoom_start=13,
    min_zoom=12,      # Prevents zooming out to see the whole world
    max_zoom=18,      # Allows deep street-level inspection
    tiles="CartoDB positron", # The "Google-like" clean professional style
    max_bounds=True   # This is the "Lock"
)

# 4. Apply the Boundary Lock
# This prevents the user from panning away from Casablanca
m.fit_bounds([[C_LAT_MIN, C_LON_MIN], [C_LAT_MAX, C_LON_MAX]])

# 5. Add the Taxis (Heatmap for better performance)
HeatMap(points, radius=8, blur=10).add_to(m)

# 6. Save and Display
m.save(r"../data/casablanca-interactive-locked.html")
m

# Save the final transformed and enriched data to MinIO

In [ ]:
final_output.write.mode("overwrite").parquet("s3a://taasim/curated/casablanca_trips_final/")

In [ ]:
import sys
print(sys.version)

In [ ]:
# df___ = spark.read.parquet("s3a://taasim/curated/casablanca_trips_final/")